In [0]:
%pip install seaborn matplotlib scikit-learn --quiet
dbutils.library.restartPython()

In [0]:
import sys
import os

# 1. Configuración de ruta dinámica
notebook_path = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_path, ".."))
FOLDER_NAME = "src" 

src_path = os.path.join(project_root, FOLDER_NAME)

# Validación de seguridad
if not os.path.exists(src_path):
    # Fallback por si el notebook no está en subcarpeta
    src_path = os.path.join(notebook_path, FOLDER_NAME)

if os.path.exists(src_path) and src_path not in sys.path:
    sys.path.append(src_path)
    print(f"✅ Ruta añadida: {src_path}")
else:
    print(f"❌ Error: No encuentro la carpeta '{FOLDER_NAME}'")

# Recarga automática de módulos (Vital para EDA iterativo)
%load_ext autoreload
%autoreload 2

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nombre_paquete.visualization import plot


In [0]:

import pandas as pd
import numpy as np

table_name = "climate_data_raw"
print(f"Leyendo datos desde la tabla Delta: {table_name}...")

try:
    # 1. Traemos los datos de Spark a Pandas
    df = spark.table(table_name).toPandas()
    
    # --- FIX DEL ERROR ARROW ---
    if 'date' in df.columns:
        print("Convirtiendo columna 'date'...")
        # 'coerce' transforma fechas inválidas en NaT (Not a Time) en lugar de dar error
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        
        # OJO: Si después de convertir sigue habiendo problemas o quedan NaT masivos,
        # verificamos si hay nulos
        nulos_fecha = df['date'].isna().sum()
        if nulos_fecha > 0:
            print(f"⚠️ Advertencia: {nulos_fecha} fechas no se pudieron leer y son NaT")

    print(f"✅ Datos cargados. Shape: {df.shape}")
    
    # 2. Ahora sí mostramos (display usa Arrow internamente)
    display(df.head())
    
except Exception as e:
    print(f"❌ Error durante la carga o conversión:")
    print(e)

In [0]:

print("--- Estadísticas Descriptivas ---")
display(df.describe().T) 

# 2. Revisión de tipos de datos
print("\n--- Tipos de Datos ---")
# df.info() imprime texto, df.dtypes te da una lista limpia
print(df.dtypes)

# 3. Conteo de Nulos (Visualmente mejorado)
print("\n--- Conteo de Nulos ---")
# Convertimos la serie a DataFrame para usar 'display' y ver una tabla bonita





In [0]:
nulos_df = pd.DataFrame(df.isnull().sum(), columns=['Total Nulos'])
# Agregamos porcentaje para ver el impacto real
nulos_df['% Nulos'] = (nulos_df['Total Nulos'] / len(df)) * 100

In [0]:
# Mostramos solo los que tienen algun nulo 
if not nulos_df[nulos_df['Total Nulos'] > 0].empty:    
    display(nulos_df[nulos_df['Total Nulos'] > 0])

In [0]:
# A. Columnas numéricas generales para histogramas y correlación
mis_cols_numericas = [
    'avg_temperature', 'humidity', 'co2_emission', 
    'energy_consumption', 'renewable_share', 
    'urban_population', 'industrial_activity_index', 'energy_price'
]

# B. Variables específicas para el gráfico de tiempo (Doble Eje)
var_tiempo_1 = 'co2_emission'
var_tiempo_2 = 'energy_consumption'

# C. Variables para el Ranking (Top 10)
cat_pais = 'country'
metrica_ranking = 'energy_consumption'

In [0]:
# 1. Distribuciones
plot.plot_numeric_distributions(df, mis_cols_numericas)

# 2. Mapa de Calor
plot.plot_correlation_heatmap(df, mis_cols_numericas)

# 3. Evolución Temporal (Específico: CO2 vs Energía)
plot.plot_dual_axis_timeseries(
    df, 
    date_col='date', 
    col1=var_tiempo_1, 
    col2=var_tiempo_2,
    label1="Emisiones CO2",
    label2="Consumo Energía"
)

# 4. Ranking de Países
plot.plot_categorical_ranking(df, cat_pais, metrica_ranking, top_n=10)

# 5. Check de integridad (Población Urbana)
plot.check_group_consistency(df, group_col='country', target_col='urban_population')